In [ ]:
import os
import numpy as np
import pandas as pd
import pyemu
import matplotlib.pyplot as plt
import flopy

In [ ]:
var_map = {
    'ph': 'pH',
    'so4': 'SO4',
    'no3': 'NO3',
    'o0': 'DO',
    'tmp': 'Temp',
    'tic': 'TIC'
}
unit_map = {
    'ph': '',
    'so4': '(mol L$^{-1}$)',
    'no3': '(mol L$^{-1}$)',
    'o0': '(mol L$^{-1}$)',
    'tic': '(mol L$^{-1}$)',
    'tmp': '($^\circ$C)',
}

limits_map = {
    'ph': [6,8],
    'so4': [0, 1.5e-3],
    'no3': [0, 5.2e-4],
    'o0': [0, 1e-3],
    'tic': [0, 0.01],
    'tmp': [0,30],
}

In [ ]:
def plot_ensemble_array_pars(td,iiter=0,casename="dizon36"):
    from matplotlib.backends.backend_pdf import PdfPages
    import matplotlib.pyplot as plt
    from matplotlib.colors import LogNorm


    sim = flopy.mf6.MFSimulation.load(sim_ws=td,verbosity_level=0)
    gwf = sim.get_model()

    pst = pyemu.Pst(os.path.join(td,f"{casename}.pst"))
    obs = pst.observation_data
    par = pst.parameter_data

    if casename=="truth":
        oe = pst.res.loc[:,["modelled"]].T.copy()
        pe = par.loc[:,["parval1"]].T.copy()
    else:
        pe = pyemu.ParameterEnsemble.from_csv(pst, os.path.join(td,f"{casename}.{iiter}.par.csv"))
        oe = pyemu.ObservationEnsemble.from_csv(pst, os.path.join(td,f"{casename}.{iiter}.obs.csv"))
    
    array_obsgrps = [i for i in obs.obgnme.unique() if "layer" in i]
    # array_obsgrps.extend(["npf.k", "sto.ss"])
    
    plot_dir = os.path.join(td,"figures")
    if os.path.exists(plot_dir)==False:
        os.mkdir(plot_dir)
    with PdfPages(os.path.join(plot_dir,"array_reals.pdf")) as pdf:
        for tag in array_obsgrps:
            kobs = obs.loc[obs.obgnme==tag].copy()
            kobs.i = kobs.i.astype(int)
            kobs.j = kobs.j.astype(int)
            kobs.sort_values(by=["i","j"],inplace=True)
            
            for real in oe.index:
                print(f"{tag}_real:{real}")

                fig,ax = plt.subplots(1,1)

                pm = flopy.plot.PlotMapView(model=gwf,ax=ax)
                ax.set_aspect("equal")
                ax.set_title(f"{tag}_real:{real}")

                a = np.zeros((gwf.dis.nrow.get_data(),gwf.dis.ncol.get_data()))
                a[:] = np.nan
                a[kobs.i,kobs.j] = oe.loc[real,kobs.obsnme].values
                # a[a <= 0] = np.nan 
                # a = np.log10(a)
                pa = pm.plot_array(a,
                                #    norm=LogNorm(vmin=np.nanmin(a), vmax=np.nanmax(a)),
                                   cmap="viridis",
                                   alpha=0.8)
                pg = pm.plot_grid()

                cb = plt.colorbar(pa)

                ppdf = pd.read_csv(os.path.join(td,f'pp.{tag}_inst0pp.dat'),header=None,sep='\s+')

                kpar = par.loc[par.pargp=="k"].copy()
                kpar[["i","j"]] = kpar[["i","j"]].astype(int)
                kpar.sort_values(by=["i","j"],inplace=True)

                #c = pe.loc[real,kpar.parnme.values].values
                ax.scatter(ppdf.iloc[:,1],ppdf.iloc[:,2],s=10,
                           c='0.5',marker=".",label="pp",
                           #c=c,cmap="viridis",
                           alpha=1,zorder=100)

                fig.tight_layout()
                pdf.savefig()
                plt.close(fig)

    return

In [ ]:
# plot_ensemble_array_pars(os.path.join('pest', 'master'),iiter=0,casename="dizon36")

# Base model

In [ ]:
ws = os.path.join("model", "reactive")
sim = flopy.mf6.MFSimulation.load(sim_ws = ws,
                                  sim_name = 'gwf', 
                                  version='mf6',
                                    exe_name='mf6',
                                    verbosity_level=0)
gwf = sim.get_model("gwf")

sout = pd.read_csv(os.path.join(ws, "sout.csv"))

from flopy.utils.gridintersect import GridIntersect

nlay = gwf.dis.nlay.get_data()
ncol = gwf.dis.ncol.get_data()
nrow = gwf.dis.nrow.get_data()

ix = GridIntersect(gwf.modelgrid)
obsloc = pd.read_csv(os.path.join("data", "obs_loc.csv"))

obs_list=[]

for obsid in obsloc.obsid.unique():
    x,y = obsloc.loc[obsloc.obsid==obsid,['x','y']].values[0]
    cellid = ix.intersect([(x,y)],shapetype="point").cellids
    if len(cellid)==0:
        continue
    else:
        cellid = cellid[0]
        obs_layer = int(obsloc.loc[obsloc.obsid==obsid,'layer'].values[0])
        obs_list.append((obsid, 'concentration', (obs_layer, cellid[0], cellid[1])))
obs_list

wells = pd.DataFrame(obs_list)
wells.columns = ['obsid', 'n', 'cell']
wells['flat_index'] = [cell[0] * (nrow * ncol) + cell[1] * ncol + cell[2] for cell in wells.cell]
wells

obs_to_index = dict(zip(wells['flat_index'], wells['obsid'].str.lower()))
obs_to_index

sout['cell'] = sout['cell'].astype(int)
sout['obsid'] = sout['cell'].map(obs_to_index)
sel = sout.dropna(subset='obsid', inplace=False).copy()
sel.columns = [i.lower() for i in sel.columns]
sel
sel['wp'] = sel['obsid'].str.extract(r'((?:wp|ip|pp)\d+)', expand=False)
sel['f']  = sel['obsid'].str.extract(r'(f\d+)',  expand=False)

sel

# Ensemble stuff

In [ ]:
md = os.path.join('pest', 'master0')
casename = 'dizon36'
pst = pyemu.Pst(os.path.join(md,f'{casename}.pst'))


oepr = pyemu.ObservationEnsemble.from_csv(pst,os.path.join(md,f"{casename}.0.obs.csv"))
noise = pyemu.ObservationEnsemble.from_csv(pst,os.path.join(md,f"{casename}.obs+noise.csv"))
noise = noise.replace(1e30, np.nan)
noise = noise.replace(0.0, np.nan)
obs=pst.observation_data

measdata = pd.read_csv(os.path.join("data", "obs_chem_cleaned.csv"))
measdata['variable'] = measdata['variable'].str.lower()
measdata['wp'] = measdata['obsid'].str.extract(r'((?:wp|ip|pp)\d+)', expand=False)
measdata['f']  = measdata['obsid'].str.extract(r'(f\d+)',  expand=False)
noise


In [ ]:
obshm = obs[obs.oname=='hm'].copy()
obshm.loc[:, 'time'] = obshm['time'].astype(float)


obshm.loc[:, 'obsid'] = obshm['obsid'].astype(str)
obshm.loc[:, 'wp']  = obshm['obsid'].str.extract(r'((?:wp|ip|pp)\d+)', expand=False)
obshm.loc[:, 'f']   = obshm['obsid'].str.extract(r'(f\d+)',  expand=False)
mask = (obshm['obgnme'] == 'cl') & (obshm.weight == 1)
obshm.loc[mask]


In [ ]:
# lets focus on f2 for now
wps = sorted(obshm['wp'].unique())[2:-1]
fs  = sorted(obshm['f'].unique())  
# fs = ['f2']
variables = sorted(obshm['variable'].unique()) 
nrow, ncol = len(fs), len(wps)

vars_to_plot = ['ph']

for var in variables:
    fig, axes = plt.subplots(
        nrow, ncol,
        figsize=(3.2 * ncol, 2.2 * nrow),
        sharex='col', sharey='row'
    )
    # axes is a 2‑D array even if any dimension is 1
    axes = np.atleast_2d(axes)

    for r, f in enumerate(fs):
        for c, wp in enumerate(wps):
            ax = axes[r, c]
            mask  = (obshm['wp'] == wp) & (obshm['f'] == f) & (obshm['variable']==var)
            obsnmes = obshm.loc[mask].sort_values('time')['obsnme'].values
            time = obshm.loc[mask].sort_values('time')['time'].values

            # meas = measdata.loc[]
            meas_pts = measdata[
                (measdata['wp'] == wp) &
                (measdata['f'] == f) &
                (measdata['variable']==var)
            ]
            ax.scatter(meas_pts['time'], meas_pts['value'], s=10, zorder=10)
            ax.plot(time, oepr.loc['base', obsnmes], color='blue', linewidth=1.5, label='base')
            [ax.scatter(time, noise.loc[i,obsnmes].values, c='red', s=1) for i in noise.index]
            [ax.plot(time, oepr.loc[i,obsnmes].values, c='0.5') for i in oepr.index]
            # Formatting
            if r == 0:                       # top row column titles
                ax.set_title(wp.upper())
            if c == 0:                       # first col ylabel per row
                ax.set_ylabel(f.upper() + f"\n{var}")
fig.tight_layout()
plt.show()

## Only plot f2 as in the paper 


In [ ]:
import matplotlib.ticker as mticker

# Fixed f
f = 'f2'

# WPs as columns
wps = sorted(obshm['wp'].unique())[2:-1]
wps.reverse()
# Variables as rows
variables = sorted(obshm['variable'].unique())
var_to_plot = ['o0', 'no3', 'so4', 'tic', 'ph']
nrow, ncol = len(var_to_plot), len(wps)

fig, axes = plt.subplots(
    nrow, ncol,
    figsize=(8, 8.5),
    sharex='col', sharey='row'
)

axes = np.atleast_2d(axes)
for r, var in enumerate(var_to_plot):
    for c, wp in enumerate(wps):
        if var == 'tmp':
            mult=1e3
        else: mult=1
        ax = axes[r, c]

        mask = (
            (obshm['wp'] == wp) &
            (obshm['f'] == f) &
            (obshm['variable'] == var)
        )
        df_masked = obshm.loc[mask].sort_values('time')
        obsnmes = df_masked['obsnme'].values
        time = df_masked['time'].values

        mask  = (sel['wp'] == wp) & (sel['f'] == f)
        lines = sel.loc[mask]
        # Measurement points
        meas_pts = measdata[
            (measdata['wp'] == wp) &
            (measdata['f'] == f) &
            (measdata['variable'] == var)
        ]
        ax.scatter(meas_pts['time'], meas_pts['value']*mult,
                   c="#E33030",
                   s=30, 
                   zorder=10)
        # Base model
        ax.plot(time, oepr.loc['base', obsnmes]*mult, color='0.5', linewidth=1.5, label='base')
        ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
        ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 2))

        # Formatting
        if r == 0:
            ax.set_title(wp.upper())
            ax.set_ylim(6,8)
        if c == 0:
            ax.set_ylabel(f"{var_map[var]} {unit_map[var]}")
        if r == nrow-1:
            ax.set_xlabel('Time (days)')
        ax.set_ylim(limits_map[var])
        ax.set_xlim(0, 875)
fig.tight_layout()
plt.savefig(os.path.join('output', 'dizon_calib.png'), 
            dpi=300,
            bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.ticker as mticker

# Fixed f
f = 'f2'

# WPs as columns
wps = sorted(obshm['wp'].unique())[2:-1]
wps.reverse()
# Variables as rows
variables = sorted(obshm['variable'].unique())
var_to_plot = ['o0', 'no3', 'so4', 'tic', 'ph', 'tmp']
nrow, ncol = len(var_to_plot), len(wps)

fig, axes = plt.subplots(
    nrow, ncol,
    figsize=(3.2 * ncol, 2.2 * nrow),
    sharex='col', sharey='row'
)

axes = np.atleast_2d(axes)
for r, var in enumerate(var_to_plot):
    for c, wp in enumerate(wps):
        if var == 'tmp':
            mult=1e3
        else: mult=1
        ax = axes[r, c]

        mask = (
            (obshm['wp'] == wp) &
            (obshm['f'] == f) &
            (obshm['variable'] == var)
        )
        df_masked = obshm.loc[mask].sort_values('time')
        obsnmes = df_masked['obsnme'].values
        time = df_masked['time'].values

        mask  = (sel['wp'] == wp) & (sel['f'] == f)
        lines = sel.loc[mask]
        # Measurement points
        meas_pts = measdata[
            (measdata['wp'] == wp) &
            (measdata['f'] == f) &
            (measdata['variable'] == var)
        ]
        ax.scatter(meas_pts['time'], meas_pts['value']*mult,
                   c="#E33030",
                   s=30, 
                   zorder=10)
        # ax.plot(lines.time, lines[var], c='blue')
        # Base model
        ax.plot(time, oepr.loc['base', obsnmes]*mult, color='0.5', linewidth=1.5, label='base')
        [ax.plot(time, oepr.loc[i,obsnmes].values*mult, c='0.5') for i in oepr.index]

        ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
        ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 2))

        # Formatting
        if r == 0:
            ax.set_title(wp.upper())
            ax.set_ylim(6,8)
        if c == 0:
            ax.set_ylabel(f"{var_map[var]} {unit_map[var]}")
        if r == nrow-1:
            ax.set_xlabel('Time (days)')
        ax.set_ylim(limits_map[var])
fig.tight_layout()
plt.savefig(os.path.join('output', 'dizon_ensem.png'), 
            dpi=300,
            bbox_inches='tight')
plt.show()


## Percentile plots 

In [ ]:
# lets focus on f2 for now
wps = sorted(obshm['wp'].unique())[2:-1]
fs  = sorted(obshm['f'].unique())  
variables = sorted(obshm['variable'].unique()) 
nrow, ncol = len(fs), len(wps)

vars_to_plot = ['ph']

for var in variables:
    fig, axes = plt.subplots(
        nrow, ncol,
        figsize=(3.2 * ncol, 2.2 * nrow),
        sharex='col', sharey='row'
    )
    # axes is a 2‑D array even if any dimension is 1
    axes = np.atleast_2d(axes)

    for r, f in enumerate(fs):
        for c, wp in enumerate(wps):
            ax = axes[r, c]
            mask  = (obshm['wp'] == wp) & (obshm['f'] == f) & (obshm['variable']==var)
            obsnmes = obshm.loc[mask].sort_values('time')['obsnme'].values
            time = obshm.loc[mask].sort_values('time')['time'].values
            time = np.asarray(time, dtype=float)

            # meas = measdata.loc[]
            meas_pts = measdata[
                (measdata['wp'] == wp) &
                (measdata['f'] == f) &
                (measdata['variable']==var)
            ]
            ax.scatter(meas_pts['time'], meas_pts['value'], s=10, zorder=10)
            # [ax.scatter(time, noise.loc[i,obsnmes].values, c='red', s=1) for i in noise.index]
            data_matrix = oepr.loc[:, obsnmes].to_numpy(dtype=float)

            # shape: (n_realizations, n_times)
            p5 = np.percentile(data_matrix, 5, axis=0)
            p95 = np.percentile(data_matrix, 95, axis=0)
            p50 = np.percentile(data_matrix, 50, axis=0)  # optional median

            ax.fill_between(time, p5, p95, color='lightblue', alpha=0.5, label='P5-P95')
            ax.plot(time, oepr.loc['base', obsnmes], color='blue', linewidth=1.5, label='base')
            # Formatting
            if r == 0:                       # top row column titles
                ax.set_title(wp.upper())
                
            if c == 0:                       # first col ylabel per row
                ax.set_ylabel(f.upper() + f"\n{var}")
fig.tight_layout()
plt.show()